### 1. 패키지 설치

In [41]:
%pip install -qU langchain langchain-openai langchain-pinecone langchain-unstructured "unstructured[docx]" python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 2. 환경변수 로드

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

### 3. 파싱 + 청킹

In [ ]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    file_path='../../files/doc/tax.docx',
    chunking_strategy='by_title',
    max_characters=1000,
    overlap=200,
    languages=['kor', 'eng'],
    include_orig_elements=False
)

docs = loader.load()

### 4. 임베딩

In [7]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large'
)

### 5. 벡터 스토어

In [4]:
from pinecone import ServerlessSpec, Pinecone

pc = Pinecone()

index_name = 'tax-docx-index'

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=3072,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )

index = pc.Index(index_name)

In [8]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index=index,
    embedding=embedding
)

In [17]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(docs))]

vector_store.add_documents(documents=docs, ids=uuids)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


['c154b9fd-d498-46fa-877a-db927167305d',
 'c70850a6-fabc-40ec-a3cb-09931d0e6584',
 '7b1d4cd7-336c-4d22-b60c-816a81449108',
 '9d6ecc03-1a1c-4450-b1b7-337e5af757ad',
 '52156804-4cbc-458b-9238-8028fc6ed001',
 '5640de61-aa90-4afd-9b24-9e452aa2253d',
 '136a9069-9509-46c7-880f-aad02e2cce5f',
 'f05c8cbd-a51a-4bf7-8fe1-9cbea68bd854',
 '74679645-44d1-4bd1-8e8b-7b9a0580d8d8',
 'e502b7cd-893b-4714-9b50-09287f02c5c7',
 '1dd61f61-7f0b-465f-9e3f-7e4ca7ccc0f5',
 'ebd11749-be53-47eb-8f7a-a927f773d203',
 '54db98b3-e7a3-4d11-a872-27ffbae24c9f',
 'db7dee85-490b-4f39-9072-54745806fe5a',
 '3fb29826-ff16-4541-b692-4d5652cab934',
 'ba90ab37-f189-4aa4-a385-58e179db31f6',
 '94e6ca67-6b53-452e-9b76-2b17c6c85e0f',
 'd8dd9b3b-33fa-4f1c-b633-dd06438e8a06',
 'b7d9437a-84e1-42cf-b1fc-8a2599df3110',
 '8d3f2d1b-7a6e-4ef5-b751-d6aa65b83517',
 '081ef0a2-f160-40f3-83be-79bccda0834d',
 '21bf66c6-e527-4818-82f2-e5b560140d20',
 'a0d62771-2f1f-434f-8b5a-a47b404ecc19',
 '608b0182-4997-4346-b6eb-f900163e4ae3',
 '1943f830-26d0-

### 6. 리트리버(검색)
- 검색 방식1. 유사도 기준 top-k

In [42]:
retriever = vector_store.as_retriever(
    search_type='similarity', # 옵션 사용하지 않으면 디폴트 similarity
    search_kwargs={'k':3}
)

In [43]:
retriever.invoke('거주자의 정의는?')

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[Document(id='30a80467-b0e9-41fe-954e-a23b0784f957', metadata={'category': 'CompositeElement', 'element_id': '8be6dd81f741b72ae43984b90a618317', 'file_directory': '../../files/doc', 'filename': 'tax.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'languages': ['kor', 'eng'], 'last_modified': '2026-06-08T20:44:05', 'page_number': 2.0, 'source': '../../files/doc/tax.docx'}, page_content='6. “1세대”란 거주자 및 그 배우자(법률상 이혼을 하였으나 생계를 같이 하는 등 사실상 이혼한 것으로 보기 어려운 관계에 있는 사람을 포함한다. 이하 이 호에서 같다)가 그들과 같은 주소 또는 거소에서 생계를 같이 하는 자[거주자 및 그 배우자의 직계존비속(그 배우자를 포함한다) 및 형제자매를 말하며, 취학, 질병의 요양, 근무상 또는 사업상의 형편으로 본래의 주소 또는 거소에서 일시 퇴거한 사람을 포함한다]와 함께 구성하는 가족단위를 말한다. 다만, 대통령령으로 정하는 경우에는 배우자가 없어도 1세대로 본다.\n\n7. “주택”이란 허가 여부나 공부(公簿)상의 용도구분과 관계없이 세대의 구성원이 독립된 주거생활을 할 수 있는 구조로서 대통령령으로 정하는 구조를 갖추어 사실상 주거용으로 사용하는 건물을 말한다. 이 경우 그 용도가 분명하지 아니하면 공부상의 용도에 따른다.\n\n8. “농지”란 논밭이나 과수원으로서 지적공부(地籍公簿)의 지목과 관계없이 실제로 경작에 사용되는 토지를 말한다. 이 경우 농지의 경영에 직접 필요한 농막, 퇴비사, 양수장, 지소(池沼), 농도(農道) 및 수로(水路) 

- 검색 방식2. MMR 기준

In [44]:
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k':3, 'fetch_k':10, 'lambda_mult':0.3}
)

In [45]:
retriever.invoke('거주자의 정의는?')

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[Document(metadata={'category': 'CompositeElement', 'element_id': '8be6dd81f741b72ae43984b90a618317', 'file_directory': '../../files/doc', 'filename': 'tax.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'languages': ['kor', 'eng'], 'last_modified': '2026-06-08T20:44:05', 'page_number': 2.0, 'source': '../../files/doc/tax.docx'}, page_content='6. “1세대”란 거주자 및 그 배우자(법률상 이혼을 하였으나 생계를 같이 하는 등 사실상 이혼한 것으로 보기 어려운 관계에 있는 사람을 포함한다. 이하 이 호에서 같다)가 그들과 같은 주소 또는 거소에서 생계를 같이 하는 자[거주자 및 그 배우자의 직계존비속(그 배우자를 포함한다) 및 형제자매를 말하며, 취학, 질병의 요양, 근무상 또는 사업상의 형편으로 본래의 주소 또는 거소에서 일시 퇴거한 사람을 포함한다]와 함께 구성하는 가족단위를 말한다. 다만, 대통령령으로 정하는 경우에는 배우자가 없어도 1세대로 본다.\n\n7. “주택”이란 허가 여부나 공부(公簿)상의 용도구분과 관계없이 세대의 구성원이 독립된 주거생활을 할 수 있는 구조로서 대통령령으로 정하는 구조를 갖추어 사실상 주거용으로 사용하는 건물을 말한다. 이 경우 그 용도가 분명하지 아니하면 공부상의 용도에 따른다.\n\n8. “농지”란 논밭이나 과수원으로서 지적공부(地籍公簿)의 지목과 관계없이 실제로 경작에 사용되는 토지를 말한다. 이 경우 농지의 경영에 직접 필요한 농막, 퇴비사, 양수장, 지소(池沼), 농도(農道) 및 수로(水路) 등에 사용되는 토지를 포함한다.\n\n9. “조합원입주권”이란 「도시 및 주거

- 검색 방식3. 임계값(score_threshold) 기준

In [46]:
retriever = vector_store.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={'k':3, 'score_threshold':0.3}
)

In [47]:
retriever.invoke('거주자의 정의는?')

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[Document(id='30a80467-b0e9-41fe-954e-a23b0784f957', metadata={'category': 'CompositeElement', 'element_id': '8be6dd81f741b72ae43984b90a618317', 'file_directory': '../../files/doc', 'filename': 'tax.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'languages': ['kor', 'eng'], 'last_modified': '2026-06-08T20:44:05', 'page_number': 2.0, 'source': '../../files/doc/tax.docx'}, page_content='6. “1세대”란 거주자 및 그 배우자(법률상 이혼을 하였으나 생계를 같이 하는 등 사실상 이혼한 것으로 보기 어려운 관계에 있는 사람을 포함한다. 이하 이 호에서 같다)가 그들과 같은 주소 또는 거소에서 생계를 같이 하는 자[거주자 및 그 배우자의 직계존비속(그 배우자를 포함한다) 및 형제자매를 말하며, 취학, 질병의 요양, 근무상 또는 사업상의 형편으로 본래의 주소 또는 거소에서 일시 퇴거한 사람을 포함한다]와 함께 구성하는 가족단위를 말한다. 다만, 대통령령으로 정하는 경우에는 배우자가 없어도 1세대로 본다.\n\n7. “주택”이란 허가 여부나 공부(公簿)상의 용도구분과 관계없이 세대의 구성원이 독립된 주거생활을 할 수 있는 구조로서 대통령령으로 정하는 구조를 갖추어 사실상 주거용으로 사용하는 건물을 말한다. 이 경우 그 용도가 분명하지 아니하면 공부상의 용도에 따른다.\n\n8. “농지”란 논밭이나 과수원으로서 지적공부(地籍公簿)의 지목과 관계없이 실제로 경작에 사용되는 토지를 말한다. 이 경우 농지의 경영에 직접 필요한 농막, 퇴비사, 양수장, 지소(池沼), 농도(農道) 및 수로(水路) 

- 배치 검색

In [48]:
retriever.batch([
    '거주자의 정의는?',
    '배당소득 계산법'
])

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[[Document(id='30a80467-b0e9-41fe-954e-a23b0784f957', metadata={'category': 'CompositeElement', 'element_id': '8be6dd81f741b72ae43984b90a618317', 'file_directory': '../../files/doc', 'filename': 'tax.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'languages': ['kor', 'eng'], 'last_modified': '2026-06-08T20:44:05', 'page_number': 2.0, 'source': '../../files/doc/tax.docx'}, page_content='6. “1세대”란 거주자 및 그 배우자(법률상 이혼을 하였으나 생계를 같이 하는 등 사실상 이혼한 것으로 보기 어려운 관계에 있는 사람을 포함한다. 이하 이 호에서 같다)가 그들과 같은 주소 또는 거소에서 생계를 같이 하는 자[거주자 및 그 배우자의 직계존비속(그 배우자를 포함한다) 및 형제자매를 말하며, 취학, 질병의 요양, 근무상 또는 사업상의 형편으로 본래의 주소 또는 거소에서 일시 퇴거한 사람을 포함한다]와 함께 구성하는 가족단위를 말한다. 다만, 대통령령으로 정하는 경우에는 배우자가 없어도 1세대로 본다.\n\n7. “주택”이란 허가 여부나 공부(公簿)상의 용도구분과 관계없이 세대의 구성원이 독립된 주거생활을 할 수 있는 구조로서 대통령령으로 정하는 구조를 갖추어 사실상 주거용으로 사용하는 건물을 말한다. 이 경우 그 용도가 분명하지 아니하면 공부상의 용도에 따른다.\n\n8. “농지”란 논밭이나 과수원으로서 지적공부(地籍公簿)의 지목과 관계없이 실제로 경작에 사용되는 토지를 말한다. 이 경우 농지의 경영에 직접 필요한 농막, 퇴비사, 양수장, 지소(池沼), 농도(農道) 및 수로(水路)